In [6]:
!pip -q install -U "transformers>=4.50.0" "accelerate>=0.35.0" "peft>=0.13.0" safetensors huggingface_hub

In [1]:
from google.colab import userdata
from huggingface_hub import login

hfToken = userdata.get('HF_TOKEN')
if not hfToken:
    raise ValueError("Secret not found. Check the secret name in Colab (left sidebar → Secrets).")

login(token=hfToken)
print("✅ Logged into Hugging Face")

✅ Logged into Hugging Face


In [2]:
baseId = "Qwen/Qwen2.5-7B-Instruct"
adapterId = "rvinay73/autotrain-xck6s-b8882"
outDir = "gate-shape-compiler"

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1) Load the tokenizer from the ADAPTER repo (this is the important part)
tokenizer = AutoTokenizer.from_pretrained(adapterId, use_fast=True)

# 2) Load base model on CPU
base = AutoModelForCausalLM.from_pretrained(
    baseId,
    torch_dtype=torch.float16,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True,
)

# 3) Resize base embeddings to match adapter tokenizer vocab
base.resize_token_embeddings(len(tokenizer))

# 4) Now load adapter (will match shapes)
model = PeftModel.from_pretrained(
    base,
    adapterId,
    device_map={"": "cpu"},
)

# 5) Merge and save
model = model.merge_and_unload()
model.save_pretrained(outDir, safe_serialization=True)
tokenizer.save_pretrained(outDir)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/798 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('gate-shape-compiler/tokenizer_config.json',
 'gate-shape-compiler/chat_template.jinja',
 'gate-shape-compiler/tokenizer.json')

In [4]:
from huggingface_hub import HfApi

api = HfApi()
outputRepo = "rvinay73/gate-shape-compiler"

api.create_repo(repo_id=outputRepo, private=True, exist_ok=True)
api.upload_folder(folder_path=outDir, repo_id=outputRepo)

info = api.repo_info(outputRepo)
print("✅ Uploaded to:", outputRepo)
print("🔒 Private:", info.private)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e-compiler/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

  ...ompiler/model.safetensors:   0%|          | 32.0MB / 15.2GB            

✅ Uploaded to: rvinay73/gate-shape-compiler
🔒 Private: True
